In [ ]:
import numpy as np
from pathlib import Path


## 1. CONFIG

In [ ]:
# Set BASE to the root of your dataset directory.
BASE = Path("data")            # <-- change to your data root
JOINTS_DIR = BASE / "3djoints"                    # input: walking_joints.npz
INSOLES_DIR = BASE / "insoles"                    # input: walking_insoles.npz
PREPROCESSED_DIR = BASE / "insoles_preprocessed"  # output: walking_insoles_preprocessed.npz
SELECTED_JOINTS_DIR = BASE / "joints_selected"    # output: walking_joints_selected.npz
OUTPUT_DIR = BASE / "synced"                      # output: walking_synced.npz

TARGET_FPS = 16
MAX_DELTA = 150          # ms - max allowed time gap between matched frames
FORCE_REDO = True       # True -> always re-run and overwrite
                         # False -> skip files that already exist

# --- Preprocessing & Visualization ---
SIGMA_SENSOR = 0.5        # sigma in sensor units (1.5 x inter-sensor spacing)
UPSCALE_FACTOR = 2    # spatial upscale factor
SHOW_MASKED_AS_GRAY = False
MASK_OUTSIDE_TO_NAN = False

WANTED_JOINTS = [
    "left-hip",
    "right-hip",
    "left-knee",
    "right-knee",
    "left-heel",
    "right-heel",
    "left-big-toe-tip",
    "right-big-toe-tip",
]

CONNECTIONS = [
    ("left-hip", "left-knee"),
    ("left-knee", "left-heel"),
    ("left-heel", "left-big-toe-tip"),

    ("right-hip", "right-knee"),
    ("right-knee", "right-heel"),
    ("right-heel", "right-big-toe-tip"),

    ("left-hip", "right-hip"),
]


ASSUME_ALREADY_SELECTED_IF_MATCH = True


The mask array is defined in row-major order matching the raw insole sensor layout. Row 0 is the first row of the array (displayed at the top with `origin='upper'`). The RIGHT mask is the horizontal mirror (`fliplr`) of the LEFT mask — no vertical flip.

## 2. FOOT MASKS

In [ ]:
import numpy as np
from scipy.ndimage import zoom, convolve1d
import matplotlib.pyplot as plt

# LEFT foot mask - raw sensor resolution (33 rows x 15 cols).
# Row 0 = top of the sensor array as physically installed.
LEFT_FOOT_MASK_RAW = np.array([
    [0,0,0,0,0,0,1,1,1,1,0,0,0,0,0],
    [0,0,0,0,0,1,1,1,1,1,1,0,0,0,0],
    [0,0,0,0,0,1,1,1,1,1,1,0,0,0,0],
    [0,0,0,0,1,1,1,1,1,1,1,1,0,0,0],
    [0,0,0,0,1,1,1,1,1,1,1,1,0,0,0],
    [0,0,0,1,1,1,1,1,1,1,1,1,0,0,0],
    [0,0,0,1,1,1,1,1,1,1,1,1,0,0,0],
    [0,0,0,1,1,1,1,1,1,1,1,1,0,0,0],
    [0,0,1,1,1,1,1,1,1,1,1,1,1,0,0],
    [0,0,1,1,1,1,1,1,1,1,1,1,1,0,0],
    [0,0,1,1,1,1,1,1,1,1,1,1,0,0,0],
    [0,0,1,1,1,1,1,1,1,1,1,1,0,0,0],
    [0,0,1,1,1,1,1,1,1,1,1,1,0,0,0],
    [0,0,1,1,1,1,1,1,1,1,1,1,0,0,0],
    [0,0,1,1,1,1,1,1,1,1,1,0,0,0,0],
    [0,0,1,1,1,1,1,1,1,1,1,0,0,0,0],
    [0,0,1,1,1,1,1,1,1,1,0,0,0,0,0],
    [0,0,0,1,1,1,1,1,1,1,0,0,0,0,0],
    [0,0,0,1,1,1,1,1,1,1,0,0,0,0,0],
    [0,0,0,1,1,1,1,1,1,1,0,0,0,0,0],
    [0,0,0,0,1,1,1,1,1,1,0,0,0,0,0],
    [0,0,0,0,1,1,1,1,1,1,0,0,0,0,0],
    [0,0,0,0,1,1,1,1,1,1,0,0,0,0,0],
    [0,0,0,0,1,1,1,1,1,1,0,0,0,0,0],
    [0,0,0,1,1,1,1,1,1,1,0,0,0,0,0],
    [0,0,0,1,1,1,1,1,1,1,1,0,0,0,0],
    [0,0,0,0,1,1,1,1,1,1,1,0,0,0,0],
    [0,0,0,0,1,1,1,1,1,1,1,0,0,0,0],
    [0,0,0,0,1,1,1,1,1,1,1,0,0,0,0],
    [0,0,0,0,1,1,1,1,1,1,1,0,0,0,0],
    [0,0,0,0,1,1,1,1,1,1,1,0,0,0,0],
    [0,0,0,0,0,1,1,1,1,1,1,0,0,0,0],
    [0,0,0,0,0,1,1,1,1,1,0,0,0,0,0],
], dtype=int)

# RIGHT foot mask = horizontal mirror of LEFT only (no vertical flip).
RIGHT_FOOT_MASK_RAW = np.fliplr(LEFT_FOOT_MASK_RAW)


## 3. GAUSSIAN / UPSCALE FUNCTIONS

In [ ]:
def _convert_sigma_units(sigma_sensor, upscale_factor, sensor_spacing_mm=7.5):
    sigma_px = sigma_sensor * upscale_factor
    sigma_mm = sigma_sensor * sensor_spacing_mm
    FWHM_sensor = 2.355 * sigma_sensor
    FWHM_mm = 2.355 * sigma_mm
    return sigma_px, sigma_mm, FWHM_sensor, FWHM_mm

def gaussian_kernel_1d(sigma):
    """1-D Gaussian kernel normalised to sum 1."""
    radius = int(np.ceil(3 * sigma))
    x = np.arange(-radius, radius + 1)
    kernel = np.exp(-0.5 * (x / sigma) ** 2)
    return kernel / np.sum(kernel)

def upscale_insole_frame(frame_2d: np.ndarray, upscale_factor: float) -> np.ndarray:
    """Bilinear upscaling of a single 2-D pressure frame."""
    return zoom(frame_2d, upscale_factor, order=1)

def upscale_binary_mask(mask_2d: np.ndarray, upscale_factor: float) -> np.ndarray:
    """Nearest-neighbour upscaling of a binary mask; returns a boolean array."""
    return zoom(mask_2d, upscale_factor, order=0) > 0.5

def gaussian_blur_nanaware_masked(
    data: np.ndarray,
    sigma_px: float,
    region_mask: np.ndarray,
) -> np.ndarray:
    """
    Separable Gaussian blur that is NaN-aware and constrained to region_mask.
    - Pixels INSIDE the mask with value 0 are treated normally (not NaN).
    - Pixels OUTSIDE the mask are forced to NaN in the output.
    """
    if data.ndim != 2:
        raise ValueError("Input data must be 2-D.")
    if data.shape != region_mask.shape:
        raise ValueError(f"Shape mismatch: data {data.shape} vs mask {region_mask.shape}")

    # Invalidate pixels outside the mask before convolution
    data_in = np.where(region_mask, data.astype(np.float64), np.nan)

    # Valid pixel mask: non-NaN AND inside region
    valid = (~np.isnan(data_in)) & region_mask

    # Replace NaN with 0 for convolution (weight-normalised later)
    data_filled = np.where(valid, data_in, 0.0)
    kernel = gaussian_kernel_1d(sigma_px)

    # Separable convolution along both axes
    blurred_data = convolve1d(
        convolve1d(data_filled, kernel, axis=1, mode='constant', cval=0.0),
        kernel, axis=0, mode='constant', cval=0.0,
    )
    blurred_mask = convolve1d(
        convolve1d(valid.astype(float), kernel, axis=1, mode='constant', cval=0.0),
        kernel, axis=0, mode='constant', cval=0.0,
    )

    result = np.full(data.shape, np.nan, dtype=np.float32)
    ok = blurred_mask > 1e-6
    result[ok] = blurred_data[ok] / blurred_mask[ok]

    # Force pixels outside the mask to NaN
    result[~region_mask] = np.nan
    return result

def preprocess_insoles(
    raw_insoles_4d: np.ndarray,
    sigma_sensor: float,
    upscale_factor: float,
) -> np.ndarray:
    """
    Upscale + NaN-aware Gaussian blur on every frame/foot.
    Input : (M, 2, 33, 15) raw pressure
    Output : (M, 2, H_up, W_up) float32
    """
    M, num_feet, H, W = raw_insoles_4d.shape
    H_up = int(H * upscale_factor)
    W_up = int(W * upscale_factor)

    processed = np.empty((M, num_feet, H_up, W_up), dtype=np.float32)
    sigma_px, *_ = _convert_sigma_units(sigma_sensor, upscale_factor)

    left_mask_up = upscale_binary_mask(LEFT_FOOT_MASK_RAW, upscale_factor)
    right_mask_up = upscale_binary_mask(RIGHT_FOOT_MASK_RAW, upscale_factor)
    upscaled_masks = [left_mask_up, right_mask_up]
    raw_masks = [LEFT_FOOT_MASK_RAW, RIGHT_FOOT_MASK_RAW]

    for i in range(M):
        for j in range(num_feet):
            foot_map = raw_insoles_4d[i, j].astype(np.float32)

            if MASK_OUTSIDE_TO_NAN:
                foot_map = np.where(raw_masks[j].astype(bool), foot_map, np.nan)

            upscaled = upscale_insole_frame(foot_map, upscale_factor)
            smoothed = gaussian_blur_nanaware_masked(upscaled, sigma_px, upscaled_masks[j])
            processed[i, j] = smoothed

    return processed


## 4. PREPROCESS-ONLY SAVE

In [ ]:
def save_preprocessed_session(
    session_name: str,
    sigma_sensor: float,
    upscale_factor: float,
) -> None:
    out_path = PREPROCESSED_DIR / f"{session_name}_insoles_preprocessed.npz"

    if not FORCE_REDO and out_path.exists():
        print(f"[{session_name}] Preprocessed insoles already exist - skipping")
        return

    insoles_path = INSOLES_DIR / f"{session_name}_insoles.npz"
    if not insoles_path.exists():
        print(f"[{session_name}] Missing insoles file: {insoles_path}")
        return

    insoles_data = np.load(insoles_path)
    raw_insoles_X = insoles_data["X"]                  # (M, 2, 33, 15)
    insoles_t = np.asarray(insoles_data["avg_timestamps"], dtype=np.float64)

    print(f"[{session_name}] Raw Insoles : {raw_insoles_X.shape} ({len(insoles_t)} frames, {insoles_t[-1]/1000:.1f}s)")
    print(
        f"[{session_name}] Preprocessing insoles "
        f"(upscale={upscale_factor}, sigma_sensor={sigma_sensor})..."
    )

    processed_insoles_X = preprocess_insoles(
        raw_insoles_X,
        sigma_sensor=sigma_sensor,
        upscale_factor=upscale_factor,
    )

    PREPROCESSED_DIR.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(
        out_path,
        insoles=processed_insoles_X,
        timestamps_ms=insoles_t,
        sigma_sensor=np.float32(sigma_sensor),
        upscale_factor=np.int32(upscale_factor),
    )
    print(f"[{session_name}] Saved preprocessed insoles -> {out_path}\n")


## 5. JOINT SELECTION

In [ ]:
def _as_str_list(names):
    return [n.decode("utf-8") if isinstance(n, bytes) else str(n) for n in names]


def _prepare_joint_array(joints: np.ndarray) -> np.ndarray:
    """
    Ensure joints are stored as (N, J, 3).
    Accepts (N, J, 3) directly and squeezes a singleton person axis if needed.
    """
    joints = np.asarray(joints)

    if joints.ndim == 4 and joints.shape[1] == 1:
        joints = joints[:, 0]

    if joints.ndim != 3 or joints.shape[-1] != 3:
        raise ValueError(f"Expected joints with shape (N, J, 3), got {joints.shape}")

    return joints


def save_selected_joints_session(session_name: str) -> None:
    out_path = SELECTED_JOINTS_DIR / f"{session_name}_joints_selected.npz"

    if not FORCE_REDO and out_path.exists():
        print(f"[{session_name}] Selected joints already exist - skipping")
        return

    joints_path = JOINTS_DIR / f"{session_name}_joints.npz"
    if not joints_path.exists():
        print(f"[{session_name}] Missing joints file: {joints_path}")
        return

    joints_data = np.load(joints_path, allow_pickle=True)
    joints = _prepare_joint_array(joints_data["joints"])
    joints_t = np.asarray(joints_data["timestamps_ms"], dtype=np.float64)

    if joints.shape[0] == 0 or len(joints_t) == 0:
        print(f"[{session_name}] Empty joints file: {joints_path}")
        return

    if "joint_names" in joints_data:
        joint_names_all = _as_str_list(joints_data["joint_names"])

        # Keep joints exactly like in the reference code:
        # build keep_indices by scanning the original joint_names order,
        # then apply the same indices to every frame.
        keep_indices = [i for i, name in enumerate(joint_names_all) if name in WANTED_JOINTS]
        selected_joint_names = np.array([joint_names_all[i] for i in keep_indices], dtype=object)
        missing = [name for name in WANTED_JOINTS if name not in joint_names_all]

        if missing:
            print(f"[{session_name}] Missing requested joints: {missing}")

        if not keep_indices:
            print(f"[{session_name}] No requested joints found in {joints_path}")
            return

        selected_joints = joints[:, keep_indices, :]
        selected_joints = normalize_orientation(selected_joints)

    else:
        if ASSUME_ALREADY_SELECTED_IF_MATCH and joints.shape[1] == len(WANTED_JOINTS):
            print(f"[{session_name}] No 'joint_names' found - assuming joints are already selected")
            selected_joints = joints
            selected_joint_names = np.array(WANTED_JOINTS, dtype=object)
        else:
            print(f"[{session_name}] Missing 'joint_names' in {joints_path} and cannot infer selection safely")
            return

    SELECTED_JOINTS_DIR.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(
        out_path,
        joints=selected_joints.astype(np.float32),
        timestamps_ms=joints_t,
        joint_names=selected_joint_names,
    )

    total_names = len(joints_data["joint_names"]) if "joint_names" in joints_data else joints.shape[1]
    print(
        f"[{session_name}] Selected joints : {selected_joints.shape} "
        f"({len(joints_t)} frames, kept {selected_joints.shape[1]}/{total_names} joints)"
    )
    print(f"[{session_name}] Kept names: {list(selected_joint_names)}")
    print(f"[{session_name}] Saved selected joints -> {out_path}\n")

def normalize_orientation(joints):
    joints = np.asarray(joints, dtype=np.float32)
    joints_norm = np.empty_like(joints)

    for i in range(len(joints)):
        pts = joints[i].copy()

        # ------------------------------------------------------------------
        # 1) Read hips
        # ------------------------------------------------------------------
        left_hip = pts[0]
        right_hip = pts[1]

        # ------------------------------------------------------------------
        # 2) Center the whole skeleton on the mid-hip
        # ------------------------------------------------------------------
        center = 0.5 * (left_hip + right_hip)
        pts = pts - center

        # Re-read centered hips
        left_hip_c = pts[0]
        right_hip_c = pts[1]

        # ------------------------------------------------------------------
        # 3) Compute hip vector in the XZ plane
        # ------------------------------------------------------------------
        # We want to rotate only around Y, so only X and Z are affected.
        hip_vec = right_hip_c - left_hip_c
        vx = float(hip_vec[0])
        vz = float(hip_vec[2])

        # If the hip vector is too small in XZ, skip rotation
        if np.hypot(vx, vz) < 1e-8:
            pts_rot = pts.copy()
        else:
            # --------------------------------------------------------------
            # 4) Compute angle theta for rotation around Y
            # --------------------------------------------------------------
            # Rotation around Y:
            #
            # x' =  cos(theta)*x + sin(theta)*z
            # y' =  y
            # z' = -sin(theta)*x + cos(theta)*z
            #
            # We want x' = 0 for the hip vector, so:
            # cos(theta)*vx + sin(theta)*vz = 0
            #
            # A valid solution is:
            # theta = atan2(-vx, vz)
            theta = np.arctan2(-vx, vz)

            c = np.cos(theta)
            s = np.sin(theta)

            R_y = np.array([
                [ c, 0.0, s],
                [0.0, 1.0, 0.0],
                [-s, 0.0, c],
            ], dtype=np.float32)

            # Apply rotation to all joints
            pts_rot = pts @ R_y.T

        # ------------------------------------------------------------------
        # 5) Enforce a consistent direction:
        #    z(right_hip) > z(left_hip)
        # ------------------------------------------------------------------

        if pts_rot[1, 2] <= pts_rot[0, 2]:
            # Mirror along Z to force the same orientation
            pts_rot[:, 2] *= -1.0

        joints_norm[i] = pts_rot

    return joints_norm


## 6. SYNC FUNCTION

In [ ]:
def sync_session(session_name: str) -> None:
    out_path = OUTPUT_DIR / f"{session_name}_synced.npz"

    if not FORCE_REDO and out_path.exists():
        print(f"[{session_name}] Already synced - skipping (set FORCE_REDO=True to override)")
        return

    selected_joints_path = SELECTED_JOINTS_DIR / f"{session_name}_joints_selected.npz"
    if not selected_joints_path.exists():
        print(f"[{session_name}] Missing selected joints file: {selected_joints_path}")
        return

    selected_joints_data = np.load(selected_joints_path, allow_pickle=True)
    joints = np.asarray(selected_joints_data["joints"], dtype=np.float32)
    joints_t = np.asarray(selected_joints_data["timestamps_ms"], dtype=np.float64)
    joint_names = _as_str_list(selected_joints_data["joint_names"])

    preprocessed_path = PREPROCESSED_DIR / f"{session_name}_insoles_preprocessed.npz"
    if not preprocessed_path.exists():
        print(f"[{session_name}] Missing preprocessed insoles file: {preprocessed_path}")
        return

    preprocessed_data = np.load(preprocessed_path)
    processed_insoles_X = np.asarray(preprocessed_data["insoles"], dtype=np.float32)
    insoles_t = np.asarray(preprocessed_data["timestamps_ms"], dtype=np.float64)

    if len(joints_t) == 0 or len(insoles_t) == 0:
        print(f"[{session_name}] Empty timestamps - cannot sync")
        return

    print(f"[{session_name}] Selected joints : {joints.shape} ({len(joints_t)} frames, {joints_t[-1]/1000:.1f}s)")
    print(f"[{session_name}] Preprocessed Insoles : {processed_insoles_X.shape} ({len(insoles_t)} frames, {insoles_t[-1]/1000:.1f}s)")

    idx = np.searchsorted(insoles_t, joints_t, side="left")
    idx_before = idx - 1
    idx_after = idx

    dt_before = np.full(joints_t.shape, np.inf)
    dt_after = np.full(joints_t.shape, np.inf)

    mask_before = idx_before >= 0
    dt_before[mask_before] = np.abs(insoles_t[idx_before[mask_before]] - joints_t[mask_before])

    mask_after = idx_after < len(insoles_t)
    dt_after[mask_after] = np.abs(insoles_t[idx_after[mask_after]] - joints_t[mask_after])

    best_idx = np.where(dt_after < dt_before, idx_after, idx_before)
    best_dt = np.minimum(dt_before, dt_after)
    keep = best_dt <= MAX_DELTA

    missing_mask = ~keep
    missing_idx = np.where(missing_mask)[0]
    missing_t = joints_t[missing_mask]

    print(f"[{session_name}] Missing sync frames: {len(missing_idx)}")

    if len(missing_idx) > 0:
        print(
            f"[{session_name}] Missing timestamps preview "
            f"(idx, t_ms): "
            + ", ".join(f"({i}, {t:.1f})" for i, t in zip(missing_idx[:20], missing_t[:20]))
        )

        gaps = np.diff(missing_idx)
        block_starts = np.where(np.r_[True, gaps > 1])[0]
        block_ends = np.r_[block_starts[1:] - 1, len(missing_idx) - 1]

        print(f"[{session_name}] Missing blocks: {len(block_starts)}")

        block_summaries = []
        for s, e in zip(block_starts, block_ends):
            i0 = missing_idx[s]
            i1 = missing_idx[e]
            t0 = missing_t[s]
            t1 = missing_t[e]
            length = e - s + 1
            block_summaries.append(
                f"[idx {i0}->{i1} | t {t0:.1f}->{t1:.1f} ms | n={length}]"
            )

        print(
            f"[{session_name}] Missing block summary: "
            + " ".join(block_summaries[:20])
        )

    paired_joints = joints[keep]
    paired_insoles = processed_insoles_X[best_idx[keep]]
    paired_t = joints_t[keep]
    paired_insoles_t = insoles_t[best_idx[keep]]

    print(
        f"[{session_name}] Paired: {paired_joints.shape[0]} frames "
        f"({keep.sum()}/{len(keep)}, dropped {(~keep).sum()} outside {MAX_DELTA}ms)"
    )

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(
        out_path,
        joints=paired_joints,
        joint_names=np.array(joint_names, dtype=object),
        insoles=paired_insoles,
        timestamps=paired_t,
        insoles_timestamps=paired_insoles_t,
    )
    print(f"[{session_name}] Saved -> {out_path}\n")

## 7. VISUALIZATION

In [ ]:
def visualize_random_insole_frames(
    session: str,
    num_frames: int = 10,
    show_masked_as_gray: bool = False,
    mask_outside_to_nan: bool = False,
) -> None:
    """
    Load processed insole data for a session and display random frames.
    show_masked_as_gray=False + mask_outside_to_nan=False
    -> regions outside the mask show their real value.
    show_masked_as_gray=True
    -> regions outside the mask are shown in light gray.
    mask_outside_to_nan=True (only when show_masked_as_gray=False)
    -> regions outside the mask are transparent.
    """
    synced_path = OUTPUT_DIR / f"{session}_synced.npz"
    if not synced_path.exists():
        print(f"No synced file found for {session} at {synced_path}")
        return

    data = np.load(synced_path)
    if "insoles" not in data:
        print(f"No 'insoles' key in {synced_path}.")
        return

    insoles = data["insoles"]
    K = insoles.shape[0]
    if K == 0:
        print(f"[{session}] No frames in synced data.")
        return

    indices = np.sort(np.random.choice(K, size=min(num_frames, K), replace=False))
    print(f"\nVisualizing {len(indices)} random insole frames for [{session}]...")

    H_up = insoles.shape[2]
    cur_upscale = int(round(H_up / LEFT_FOOT_MASK_RAW.shape[0]))
    left_mask_up = upscale_binary_mask(LEFT_FOOT_MASK_RAW, cur_upscale)
    right_mask_up = upscale_binary_mask(RIGHT_FOOT_MASK_RAW, cur_upscale)

    cmap = plt.cm.viridis.copy()
    if show_masked_as_gray:
        cmap.set_bad(color="lightgray")
    else:
        cmap.set_bad(alpha=0.0)

    for i, idx in enumerate(indices):
        left_data = insoles[idx, 0].copy()
        right_data = insoles[idx, 1].copy()

        if show_masked_as_gray:
            left_disp = np.ma.array(left_data, mask=~left_mask_up)
            right_disp = np.ma.array(right_data, mask=~right_mask_up)
        else:
            if mask_outside_to_nan:
                left_disp = np.ma.array(left_data, mask=np.isnan(left_data))
                right_disp = np.ma.array(right_data, mask=np.isnan(right_data))
            else:
                left_disp = np.nan_to_num(left_data, nan=0.0)
                right_disp = np.nan_to_num(right_data, nan=0.0)

        fig, axes = plt.subplots(1, 2, figsize=(12, 6))
        fig.suptitle(f"[{session}] Frame {idx} ({i+1}/{len(indices)})", fontsize=13)

        im1 = axes[0].imshow(left_disp, cmap=cmap, origin="upper")
        axes[0].set_title("Left Foot")
        plt.colorbar(im1, ax=axes[0], fraction=0.046, pad=0.04)

        im2 = axes[1].imshow(right_disp, cmap=cmap, origin="upper")
        axes[1].set_title("Right Foot")
        plt.colorbar(im2, ax=axes[1], fraction=0.046, pad=0.04)

        plt.tight_layout(rect=[0, 0.03, 1, 0.95])
        plt.show()


def visualize_random_joint_frames(session: str, num_frames: int = 6) -> None:
    selected_joints_path = SELECTED_JOINTS_DIR / f"{session}_joints_selected.npz"
    if not selected_joints_path.exists():
        print(f"No selected joints file found for {session} at {selected_joints_path}")
        return

    data = np.load(selected_joints_path, allow_pickle=True)
    joints = np.asarray(data["joints"], dtype=np.float32)
    joint_names = _as_str_list(data["joint_names"])

    if len(joints) == 0:
        print(f"[{session}] No joint frames found.")
        return

    indices = np.sort(np.random.choice(len(joints), size=min(num_frames, len(joints)), replace=False))
    name_to_idx = {name: i for i, name in enumerate(joint_names)}
    valid_connections = [(a, b) for a, b in CONNECTIONS if a in name_to_idx and b in name_to_idx]

    x_all = joints[:, :, 0]
    y_all = joints[:, :, 1]
    z_all = joints[:, :, 2]

    max_range = np.array([
        x_all.max() - x_all.min(),
        y_all.max() - y_all.min(),
        z_all.max() - z_all.min(),
    ]).max() / 2
    if max_range == 0:
        max_range = 1.0

    mid_x = (x_all.max() + x_all.min()) * 0.5
    mid_y = (y_all.max() + y_all.min()) * 0.5
    mid_z = (z_all.max() + z_all.min()) * 0.5

    n = len(indices)
    ncols = min(3, n)
    nrows = int(np.ceil(n / ncols))

    fig = plt.figure(figsize=(5 * ncols, 4.8 * nrows))
    fig.suptitle(f"[{session}] Random selected-joint frames", fontsize=14)

    for plot_idx, frame_idx in enumerate(indices, start=1):
        ax = fig.add_subplot(nrows, ncols, plot_idx, projection="3d")
        points = joints[frame_idx]

        ax.scatter(
            points[:, 0],
            points[:, 1],
            points[:, 2],
            s=40,
            color="#2ca9df",
            edgecolors="black",
            linewidth=0.4,
        )

        for joint_a, joint_b in valid_connections:
            ia = name_to_idx[joint_a]
            ib = name_to_idx[joint_b]
            ax.plot(
                [points[ia, 0], points[ib, 0]],
                [points[ia, 1], points[ib, 1]],
                [points[ia, 2], points[ib, 2]],
                color="black",
                linewidth=1.6,
            )

        ax.set_xlim(mid_x - max_range, mid_x + max_range)
        ax.set_ylim(mid_y - max_range, mid_y + max_range)
        ax.set_zlim(mid_z - max_range, mid_z + max_range)
        ax.set_box_aspect([1, 1, 1])
        ax.view_init(elev=-49, azim=-7, roll=-84)
        ax.grid(True, alpha=0.3)
        ax.set_title(f"Frame {frame_idx}", fontsize=11)
        ax.set_xlabel("X")
        ax.set_ylabel("Y")
        ax.set_zlabel("Z")

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()

## 8. RUN - auto-discover sessions

In [ ]:
sessions = sorted([
    p.stem.replace("_joints", "")
    for p in JOINTS_DIR.glob("*_joints.npz")
])

if not sessions:
    print(f"No session files found in {JOINTS_DIR}")
else:
    print(f"Found {len(sessions)} session(s): {sessions}\n")

    for session in sessions:
        save_preprocessed_session(
            session,
            sigma_sensor=SIGMA_SENSOR,
            upscale_factor=UPSCALE_FACTOR,
        )
        save_selected_joints_session(session)
        #visualize_random_joint_frames(session, 20)
        #visualize_random_insole_frames(session)
        #preview_pre_sync_timestamps(session, n=10)
        sync_session(session)
